In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio

# =====================================================================
# 1. ÖZELLİK ÇIKARICI (MEL-SPECTROGRAM)
# =====================================================================
class MelSpecExtractor(nn.Module):
    def __init__(self, sample_rate=16000, n_mels=80):
        super().__init__()
        self.mel_spec = torchaudio.transforms.MelSpectrogram(
            sample_rate=sample_rate,
            n_fft=512,
            win_length=400,
            hop_length=160,
            n_mels=n_mels
        )
        self.amplitude_to_db = torchaudio.transforms.AmplitudeToDB()

    def forward(self, waveform):
        x = self.mel_spec(waveform)
        x = self.amplitude_to_db(x)
        return x

# =====================================================================
# 2. MODEL MİMARİSİ BİLEŞENLERİ
# =====================================================================
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super(SEBlock, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)

class AttentiveStatsPooling(nn.Module):
    def __init__(self, in_dim, attention_channels=128):
        super(AttentiveStatsPooling, self).__init__()
        self.attention = nn.Sequential(
            nn.Conv1d(in_dim, attention_channels, kernel_size=1),
            nn.ReLU(),
            nn.BatchNorm1d(attention_channels),
            nn.Conv1d(attention_channels, in_dim, kernel_size=1),
            nn.Softmax(dim=2)
        )

    def forward(self, x):
        weights = self.attention(x)
        mean = torch.sum(weights * x, dim=2)
        std = torch.sqrt((torch.sum((x ** 2) * weights, dim=2) - mean ** 2).clamp(min=1e-5))
        return torch.cat((mean, std), dim=1)

class SEResNetBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(SEResNetBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.se = SEBlock(out_channels)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.se(out)
        out += self.shortcut(x)
        return F.relu(out)

# =====================================================================
# 3. ANA MODEL (SOTA DEEPFAKE DETECTOR)
# =====================================================================
class SOTADeepfakeDetector(nn.Module):
    def __init__(self, num_classes=2):
        super(SOTADeepfakeDetector, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=5, stride=2, padding=2, bias=False)
        self.bn1 = nn.BatchNorm2d(32)
        self.layer1 = SEResNetBlock(32, 64, stride=2)
        self.layer2 = SEResNetBlock(64, 128, stride=2)
        self.layer3 = SEResNetBlock(128, 256, stride=2)
        self.asp = AttentiveStatsPooling(in_dim=256)
        self.classifier = nn.Sequential(
            nn.Linear(256 * 2, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        # CMVN Normalizasyonu
        x = (x - x.mean(dim=(2, 3), keepdim=True)) / (x.std(dim=(2, 3), keepdim=True) + 1e-6)
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = torch.mean(x, dim=2)
        x = self.asp(x)
        out = self.classifier(x)
        return out


def load_deepfake_model(weights_path, device='cpu'):
    """Modeli ve özellik çıkarıcıyı başlatır ve ağırlıkları yükler."""
    model = SOTADeepfakeDetector(num_classes=2)
    model.load_state_dict(torch.load(weights_path, map_location=device))
    model.to(device)
    model.eval()  # Tahmin moduna al

    extractor = MelSpecExtractor(n_mels=80).to(device)
    return model, extractor

def analyze_audio(file_path, model, extractor, device='cpu'):
    """Verilen ses dosyasını analiz eder ve JSON/Dict formatında sonuç döner."""
    if not os.path.exists(file_path):
        return {"status": "error", "message": f"Dosya bulunamadı: {file_path}"}

    try:
        # Sesi yükle ve formatla
        waveform, sr = torchaudio.load(file_path, backend="soundfile")

        # Mono (Tek Kanal) yap
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)

        # 16kHz Örnekleme Hızına Sabitle
        if sr != 16000:
            waveform = torchaudio.functional.resample(waveform, sr, 16000)

        # 4 Saniye (64000 sample) Uzunluğa Sabitle (Kısa ise tekrarla, uzunsa kırp)
        target_length = 64000
        num_samples = waveform.shape[-1]
        if num_samples < target_length:
            repeats = (target_length // num_samples) + 1
            waveform = waveform.repeat(1, repeats)[:, :target_length]
        elif num_samples > target_length:
            waveform = waveform[:, :target_length]

        # Mel-Spectrogram çıkar [1, 1, 80, 401]
        features = extractor(waveform).unsqueeze(0).to(device)

        # Modelden geçir ve olasılıkları hesapla
        with torch.no_grad():
            out = model(features)
            probs = F.softmax(out, dim=1)
            real_prob = probs[0][0].item() * 100
            fake_prob = probs[0][1].item() * 100

        # Sonucu Dön
        return {
            "status": "success",
            "file_name": os.path.basename(file_path),
            "prediction": "FAKE" if fake_prob > 50 else "REAL",
            "real_probability": round(real_prob, 2),
            "fake_probability": round(fake_prob, 2)
        }

    except Exception as e:
        return {"status": "error", "message": str(e)}

if __name__ == "__main__":

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

    MODEL_WEIGHTS_PATH = "best_sota_heavy_aug.pth"

    print(f"Sistem başlatılıyor... (Cihaz: {DEVICE})")

    if os.path.exists(MODEL_WEIGHTS_PATH):
       
        ai_detector, mel_extractor = load_deepfake_model(MODEL_WEIGHTS_PATH, device=DEVICE)
        print("✅ Deepfake Tespit Modeli başarıyla yüklendi!")

        
        test_audio_path = "deepfake_voice.mp3"  

        if os.path.exists(test_audio_path):
            result = analyze_audio(test_audio_path, ai_detector, mel_extractor, device=DEVICE)
            print("\n--- ANALİZ SONUCU ---")
            print(result)
        else:
            print(f"\n⚠️ '{test_audio_path}' adında bir dosya yok. Kendi API yapınızda analyze_audio() fonksiyonuna doğru dosya yolunu göndermelisiniz.")

    else:
        print(f"\n❌ Hata: '{MODEL_WEIGHTS_PATH}' dosyası bulunamadı. Lütfen model ağırlıklarını ilgili klasöre koyun.")


Sistem başlatılıyor... (Cihaz: cpu)
✅ Deepfake Tespit Modeli başarıyla yüklendi!

--- ANALİZ SONUCU ---
{'status': 'success', 'file_name': 'WhatsApp Ptt 2026-08-05 at 20.55.23.ogg', 'prediction': 'REAL', 'real_probability': 99.91, 'fake_probability': 0.09}
